# PS13 - Ant Colony Optimization for Communication Networks

This notebook implements:

1. PEAS description
2. Ant Colony Optimization (ACO)
3. Dijkstra Algorithm
4. Scenario comparison

The problem models a communication network as a weighted graph:
- Routers = Nodes
- Communication Links = Edges
- Latency = Edge Weight


## Part (a) - PEAS Description

| Component | Description |
|---|---|
| Performance Measure | Minimize latency and find shortest communication path |
| Environment | Communication network represented as graph |
| Actuators | Move between routers and deposit pheromones |
| Sensors | Detect neighboring nodes, latency, pheromone levels |


## Import Required Libraries

In [ ]:
import random
import heapq

## Graph Representation

Adjacency List is used because it is memory efficient for sparse graphs.


In [ ]:
class Graph:

    def __init__(self):
        self.graph = {}

    def add_edge(self, u, v, weight):

        if u not in self.graph:
            self.graph[u] = {}

        if v not in self.graph:
            self.graph[v] = {}

        self.graph[u][v] = weight
        self.graph[v][u] = weight

## Read Input File

In [ ]:
def read_input(filename):

    g = Graph()

    with open(filename, 'r') as file:

        lines = file.readlines()

        edge_count = int(lines[0].strip())

        index = 1

        for i in range(edge_count):

            u, v, w = map(int, lines[index].split())
            g.add_edge(u, v, w)

            index += 1

        source = int(lines[index].split()[1])
        destination = int(lines[index + 1].split()[1])

    return g.graph, source, destination

## Dijkstra Algorithm

Used as deterministic shortest path baseline.


In [ ]:
def dijkstra(graph, source, destination):

    pq = []
    heapq.heappush(pq, (0, source, []))

    visited = set()

    while pq:

        cost, node, path = heapq.heappop(pq)

        if node in visited:
            continue

        visited.add(node)

        path = path + [node]

        if node == destination:
            return path, cost

        for neighbor in graph[node]:

            if neighbor not in visited:

                new_cost = cost + graph[node][neighbor]

                heapq.heappush(
                    pq,
                    (new_cost, neighbor, path)
                )

    return None, float('inf')

## Ant Colony Optimization

In [ ]:
class AntColonyOptimization:

    def __init__(self,
                 graph,
                 ants,
                 alpha,
                 beta,
                 evaporation,
                 iterations=50):

        self.graph = graph
        self.ants = ants
        self.alpha = alpha
        self.beta = beta
        self.evaporation = evaporation
        self.iterations = iterations

        self.pheromone = {}

        for u in graph:
            for v in graph[u]:
                self.pheromone[(u, v)] = 1.0


    def path_cost(self, path):

        total = 0

        for i in range(len(path) - 1):

            u = path[i]
            v = path[i + 1]

            total += self.graph[u][v]

        return total


    def choose_next_node(self, current, visited):

        neighbors = []
        probabilities = []

        for neighbor in self.graph[current]:

            if neighbor not in visited:

                pheromone = self.pheromone[(current, neighbor)] ** self.alpha

                heuristic = (1 / self.graph[current][neighbor]) ** self.beta

                value = pheromone * heuristic

                neighbors.append(neighbor)
                probabilities.append(value)

        if not neighbors:
            return None

        total = sum(probabilities)

        probabilities = [p / total for p in probabilities]

        return random.choices(neighbors, probabilities)[0]


    def construct_path(self, source, destination):

        current = source

        path = [current]

        visited = set()
        visited.add(current)

        while current != destination:

            next_node = self.choose_next_node(current, visited)

            if next_node is None:
                return None

            path.append(next_node)
            visited.add(next_node)

            current = next_node

        return path


    def update_pheromones(self, all_paths):

        # Evaporation

        for edge in self.pheromone:
            self.pheromone[edge] *= (1 - self.evaporation)


        # Deposit new pheromone

        for path, cost in all_paths:

            if path is None:
                continue

            deposit = 1 / cost

            for i in range(len(path) - 1):

                u = path[i]
                v = path[i + 1]

                self.pheromone[(u, v)] += deposit
                self.pheromone[(v, u)] += deposit


    def run(self, source, destination):

        best_path = None
        best_cost = float('inf')

        for iteration in range(self.iterations):

            all_paths = []

            for ant in range(self.ants):

                path = self.construct_path(source, destination)

                if path is None:
                    continue

                cost = self.path_cost(path)

                all_paths.append((path, cost))

                if cost < best_cost:
                    best_cost = cost
                    best_path = path

            self.update_pheromones(all_paths)

        return best_path, best_cost

## Create Sample Input File

In [ ]:
sample_input = '''7
0 1 2
0 2 4
1 2 1
1 3 7
2 4 3
3 4 1
0 4 2
SOURCE 0
DESTINATION 4
'''

with open('inputPS13.txt', 'w') as file:
    file.write(sample_input)

print('Input file created successfully')

## Run Scenario 1

In [ ]:
graph, source, destination = read_input('inputPS13.txt')

aco1 = AntColonyOptimization(
    graph=graph,
    ants=10,
    alpha=1.0,
    beta=2,
    evaporation=0.5,
    iterations=50
)

result1 = aco1.run(source, destination)

print('Scenario 1')
print('Best Path:', result1[0])
print('Minimum Latency:', result1[1])

## Run Scenario 2

In [ ]:
aco2 = AntColonyOptimization(
    graph=graph,
    ants=10,
    alpha=2.5,
    beta=1.0,
    evaporation=0.3,
    iterations=50
)

result2 = aco2.run(source, destination)

print('Scenario 2')
print('Best Path:', result2[0])
print('Minimum Latency:', result2[1])

## Run Dijkstra Algorithm

In [ ]:
dijkstra_result = dijkstra(
    graph,
    source,
    destination
)

print('Dijkstra Result')
print('Best Path:', dijkstra_result[0])
print('Minimum Latency:', dijkstra_result[1])

## Write Output File

In [ ]:
with open('outputPS13.txt', 'w') as file:

    file.write('=== Scenario 1 ===\n')
    file.write(f'Best Path: {result1[0]}\n')
    file.write(f'Minimum Latency: {result1[1]}\n\n')

    file.write('=== Scenario 2 ===\n')
    file.write(f'Best Path: {result2[0]}\n')
    file.write(f'Minimum Latency: {result2[1]}\n\n')

    file.write('=== Dijkstra Algorithm ===\n')
    file.write(f'Best Path: {dijkstra_result[0]}\n')
    file.write(f'Minimum Latency: {dijkstra_result[1]}\n')

print('Output file generated successfully')

## Observations

### Scenario 1
- Higher beta gives more importance to distance
- Ants prefer shorter paths
- More exploration occurs

### Scenario 2
- Higher alpha gives more importance to pheromones
- Faster convergence happens
- Ants follow successful paths aggressively

### Comparison with Dijkstra
- Dijkstra always gives shortest path
- ACO is probabilistic
- ACO is useful in adaptive routing environments
